In [1]:
"""
의학 텍스트 분류기와 프롬프트를 통합한 파일 (GPT API 형식)
"""
import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
from tenacity import retry, stop_after_attempt, wait_exponential
import re
from prompts.medical_prompts import CCPrompts
from prompts.medical_prompts import TreatmentPrompts
from prompts.medical_prompts import TherapyPrompts
# from prompts.medical_prompts import PresentIllnessPrompts

from prompts.PI_prompts import PresentIllnessPrompts_ver2
from prompts.medical_prompts import NumericPrompts

from rate_limiter.rate_limiter import RateLimiter

from tqdm.asyncio import tqdm as tqdm_asyncio

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_excel('../../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../../data/info.csv')
api_key = api.loc[1][1]

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_40160/962184981.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  api_key = api.loc[1][1]


In [4]:
df.columns = df.columns.str.strip()
df = df[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI']]

In [11]:
df = df.sample(2)

In [81]:
class Config:
    # 실제 OpenAI API 키로 교체하거나 환경변수로부터 로드하세요.
    API_KEY = api_key
    # MODEL_NAME = "gpt-4o"
    MODEL_NAME = "o3-mini"
    MAX_TOKENS = 4096
    TEMPERATURE = 0
    BATCH_SIZE = 100
    SEMAPHORE_LIMIT = 5
    MAX_RETRIES = 4
    CHECKPOINT_DIR = "checkpoints"
    LOG_FILE = "medical_classifier.log"
    # RPM_LIMIT = 4000  # 실제 한도보다 약간 낮게 설정
    # TPM_LIMIT = 3800000  # 실제 한도보다 약간 낮게 설정
    
    # o1 모델 API 제한 반영 (약간의 여유를 둠)
    RPM_LIMIT = 4800  # 5,000 RPM
    TPM_LIMIT = 3800000  # 4,000,000 TPM
    TPD_LIMIT = 38000000  # 40,000,000 TPD
    INDEX_COLUMNS = ['환자번호', '날짜']

#############################################
# 로깅 설정 함수
#############################################

def setup_logging(log_file=Config.LOG_FILE):
    """로깅 설정을 초기화하는 함수"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

# 초기 로거 생성 (설정은 아직 적용되지 않음)
logger = logging.getLogger(__name__)


#############################################
# 체크포인트 관리 클래스
#############################################

class CheckpointManager:
    """Checkpoint management class"""
    def __init__(self, checkpoint_dir: str = Config.CHECKPOINT_DIR):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(self.checkpoint_dir, exist_ok=True)
    
    def get_checkpoint_path(self, column: str) -> str:
        safe_column = column.replace("/", "_").replace("\\", "_")
        return os.path.join(self.checkpoint_dir, f"{safe_column}_checkpoint.parquet")

    # CheckpointManager 메서드 수정 (문자열 인덱스 처리)
    def save_checkpoint(self, df: pd.DataFrame, column: str) -> None:
        try:
            # 인덱스 정보 로깅
            logger.info(f"체크포인트 저장 전 인덱스: {df.index.tolist()[:5]}")
            
            # 중요: 원본 인덱스 값으로 인덱스 설정
            if 'orig_index' in df.columns:
                # 인덱스 설정 전 'orig_index' 값 별도 저장
                orig_index_values = df['orig_index'].tolist()
                logger.info(f"orig_index 값: {orig_index_values[:5]}")
                
                # 'orig_index' 값으로 인덱스 설정
                df = df.set_index('orig_index')
                logger.info(f"인덱스 설정 후: {df.index.tolist()[:5]}")
            
            # 파일 저장
            df.to_parquet(self.get_checkpoint_path(column), index=True)
            logger.info(f"Checkpoint saved for column {column}")
        except Exception as e:
            logger.error(f"Failed to save checkpoint for {column}: {str(e)}")


    def load_checkpoint(self, column: str) -> Optional[pd.DataFrame]:
        path = self.get_checkpoint_path(column)
        if os.path.exists(path):
            try:
                # 인덱스를 명시적으로 로드하도록 설정
                df = pd.read_parquet(path)
                logger.info(f"Loaded checkpoint with index values: {df.index.tolist()[:5]}...")
                return df
            except Exception as e:
                logger.error(f"Failed to load checkpoint for {column}: {str(e)}")
        return None


#############################################
# 메디컬 텍스트 분류기 클래스 (GPT API 사용)
#############################################

class MedicalTextClassifier:
    """의학 텍스트 분류기 클래스"""
    
    def __init__(self, api_key: str, config=None):
        """초기화"""
        self.config = config if config is not None else Config
        
        # API 키 설정 방식 수정
        # 방법 1: 클라이언트 초기화 시 직접 API 키 설정
        self.client = openai.OpenAI(api_key=api_key)  
        
        # 방법 2: 환경 변수로 API 키 설정 (추가적인 방법)
        import os
        os.environ["OPENAI_API_KEY"] = api_key
        self.config = config if config is not None else Config
        self.client = openai.OpenAI(api_key=api_key)  # 클라이언트 객체 생성 방식 변경
        self.semaphore = asyncio.Semaphore(self.config.SEMAPHORE_LIMIT)
        self.checkpoint = CheckpointManager(self.config.CHECKPOINT_DIR)
        # Rate Limiter 객체 추가
        self.rate_limiter = RateLimiter(
            rpm_limit=self.config.RPM_LIMIT,
            tpm_limit=self.config.TPM_LIMIT
        )
        # 분류기 메소드 맵핑
        self.classifiers = {
            'CC': self._classify_cc,
            '약': self._classify_medication,
            '장치': self._classify_device,
            '습관': self._classify_habit,
            '찜질': self._classify_hot_pack,
            '마사지, 스트레칭': self._classify_massage,
            # 'PI': self._classify_present_illness,
        }
    
    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """모든 컬럼 처리 (원본 인덱스 보존)"""
        original_shape = df.shape
        processed_cols = 0
        
        # 디버깅: 원본 인덱스 확인
        logger.info(f"원본 DataFrame 인덱스 타입: {df.index.dtype}")
        logger.info(f"원본 DataFrame 인덱스 샘플: {df.index.tolist()[:5]}")
        
        # 원본 인덱스 저장 및 리셋
        original_index = df.index.copy()
        original_df = df.reset_index().copy()  # 인덱스를 컬럼으로 변환하여 복사
        
        # 디버깅: 리셋 후 인덱스 확인
        logger.info(f"리셋 후 DataFrame 인덱스 샘플: {original_df.index.tolist()[:5]}")
        logger.info(f"인덱스 컬럼 값 샘플: {original_df['index'].tolist()[:5]}")
        
        # 원본 인덱스를 문자열로 보존
        original_df['orig_index'] = original_index.map(str)
        
        # 디버깅: orig_index 확인
        logger.info(f"orig_index 샘플: {original_df['orig_index'].tolist()[:5]}")
        """ 로깅 """
        original_shape = df.shape
        processed_cols = 0
        
        # 원본 인덱스 저장 및 리셋
        original_index = df.index.copy()
        original_df = df.reset_index().copy()  # 인덱스를 컬럼으로 변환하여 복사
        
        # 원본 인덱스를 문자열로 보존
        original_df['orig_index'] = original_index.map(str)
        
        # 결과를 저장할 데이터프레임 초기화 (원본과 동일한 인덱스 사용)
        result_df = original_df.copy()
        
        for column in self.classifiers.keys():
            if column in df.columns:
                logger.info(f"Processing column: {column}")
                
                # 원본 인덱스 정보가 포함된 데이터프레임 전달
                column_results = await self._process_column_with_checkpoint(result_df, column)
                
                if column_results is not None and not column_results.empty:
                    # 결과 데이터프레임에 매핑 (orig_index를 기준으로)
                    # 주의: column_results의 인덱스는 이미 orig_index임
                    
                    # column_results의 모든 열을 한 번에 안전하게 병합
                    try:
                        # 병합할 때 오류 처리 추가
                        derived_cols = [col for col in column_results.columns if col.startswith(f"{column}_")]
                        if derived_cols:
                            logger.info(f"Joining {len(derived_cols)} columns from results to main dataframe")
                            # 열을 하나씩 추가
                            for derived_col in derived_cols:
                                # Series로 변환하여 병합
                                series_to_join = column_results[derived_col]
                                result_df[derived_col] = result_df['orig_index'].map(
                                    series_to_join.to_dict()
                                )
                    except Exception as e:
                        logger.error(f"Error joining columns for {column}: {str(e)}")
                        # 열을 하나씩 안전하게 병합
                        for derived_col in column_results.columns:
                            if derived_col.startswith(f"{column}_"):
                                try:
                                    # 안전하게 하나씩 매핑
                                    result_df[derived_col] = result_df['orig_index'].map(
                                        column_results[derived_col].to_dict()
                                    )
                                except Exception as e2:
                                    logger.error(f"Error adding column {derived_col}: {str(e2)}")
                    
                    # 로깅
                    derived_cols = [col for col in column_results.columns 
                                    if col.startswith(f"{column}_")]
                    logger.info(f"Column {column} generated {len(derived_cols)} derived columns: {derived_cols}")
                    
                    for derived_col in derived_cols:
                        non_empty_count = result_df[derived_col].notna().sum()
                        logger.info(f"Column {derived_col} has {non_empty_count} non-empty values")
                
                processed_cols += 1
        
        # 처리 완료 후 원래 인덱스로 복원
        # 'orig_index' 컬럼과 기존 인덱스 컬럼 삭제
        if 'index' in result_df.columns:
            result_df.drop('index', axis=1, inplace=True, errors='ignore')
        
        # 원본 인덱스로 변환 전 orig_index 컬럼 삭제
        if 'orig_index' in result_df.columns:
            result_df.drop('orig_index', axis=1, inplace=True)
        
        # 원본 인덱스 복원
        result_df.index = original_index
        
        logger.info(f"Original DataFrame shape: {original_shape}, Processed columns: {processed_cols}")
        return result_df

    async def _process_column_with_checkpoint(self, df: pd.DataFrame, column: str) -> pd.DataFrame:
        """체크포인트를 사용한 컬럼 처리 (원본 인덱스 보존)"""
        try:
            # 체크포인트 확인
            checkpoint_df = self.checkpoint.load_checkpoint(column)
            if checkpoint_df is not None:
                logger.info(f"Resumed from checkpoint for {column}")
                return checkpoint_df

            # 공백이거나 길이가 0인 레코드 제외
            mask = df[column].notna() & (df[column].astype(str).str.strip().str.len() > 0)
            
            if not mask.any():
                logger.info(f"No valid text entries found in column {column}")
                return pd.DataFrame()

            # 필터링된 텍스트와 인덱스 쌍 생성
            filtered_df = df.loc[mask].copy()
            
            logger.info(f"loggggggggggggggg index :: filtered_df: {filtered_df}")
            
            # 원본 인덱스를 문자열로 보존
            filtered_df['orig_index'] = filtered_df.index.map(str)
            
            logger.info(f"loggggggggggggggg index :: filtered_df: {filtered_df}")
            # 인덱스와 텍스트 쌍 생성
            texts_with_idx = [(idx, text, orig_idx) for idx, text, orig_idx in 
                            zip(filtered_df.index, filtered_df[column], filtered_df['orig_index'])]
            
            logger.info(f"Processing {len(texts_with_idx)} valid records for column {column}")
            
            # API 호출 및 결과 처리 (원본 인덱스 포함)
            results = await self._safe_process_batches(
                texts=[text for _, text, _ in texts_with_idx],
                original_indices=[idx for idx, _, _ in texts_with_idx],
                orig_indices=[orig_idx for _, _, orig_idx in texts_with_idx],
                classifier=self.classifiers[column],
                column=column
            )

            # 결과 처리
            if results:
                # 결과를 데이터프레임으로 변환
                results_df = pd.DataFrame(results)
                logger.info(f"loggggggggggggggg 2222 index :: results_df: {results_df}")
                # 'orig_index'를 인덱스로 설정 (이것이 원본 인덱스를 보존하는 핵심)
                if 'orig_index' in results_df.columns:
                    results_df.set_index('orig_index', inplace=True)
                
                # 컬럼명 변경: 'index'를 유지하고 나머지에만 접두사 추가
                new_columns = []
                for col in results_df.columns:
                    if col == 'index':
                        new_columns.append(col)  # 'index' 컬럼은 그대로 유지
                    else:
                        new_columns.append(f"{column}_{col}")  # 나머지 컬럼에만 접두사 추가
                
                # 디버깅을 위한 로그 추가
                logger.info(f"Original columns: {list(results_df.columns)}")
                logger.info(f"New columns: {new_columns}")
                
                # 컬럼 수 확인
                if len(new_columns) == len(results_df.columns):
                    results_df.columns = new_columns
                else:
                    logger.error(f"Column length mismatch: {len(results_df.columns)} vs {len(new_columns)}")
                    # 개별 컬럼 이름 변경으로 전환
                    for i, (old_col, new_col) in enumerate(zip(results_df.columns, new_columns)):
                        results_df = results_df.rename(columns={old_col: new_col})
                
                # 데이터프레임 저장 및 반환
                self.checkpoint.save_checkpoint(results_df, column)
                return pd.DataFrame(results_df)

            return pd.DataFrame()

        except Exception as e:
            logger.error(f"Critical error processing {column}: {str(e)}")
            raise
        
    async def _safe_process_batches(self, texts: List[str], original_indices: List[int],
                                orig_indices: List[str], classifier, column: str) -> List[Dict]:
        """비동기로 텍스트 배치 처리"""
        results = []

        for i, (text, idx, orig_idx) in enumerate(zip(texts, original_indices, orig_indices)):
            try:
                # 유효하지 않은 텍스트 건너뛰기
                if text is None or len(str(text).strip()) == 0:
                    logger.info(f"Skipping invalid text at index {idx} (None or empty)")
                    results.append({"index": idx, "orig_index": orig_idx})
                    continue
                    
                logger.info(f"단일 텍스트 처리 중 {i+1}/{len(texts)} (인덱스 {idx}, 원본 인덱스 {orig_idx})")
                
                # API 호출 (원본 인덱스 전달) - 비동기 함수 호출
                batch_result = await self._process_with_retry(classifier, [text], [idx], [orig_idx])
                
                if batch_result:
                    # 여기서 원본 인덱스 값을 배치 결과에 추가해야 함
                    for item in batch_result:
                        # 원본 인덱스 값으로 설정 (문자열)
                        item['orig_index'] = orig_idx  # 이 부분이 중요!
                    
                    results.extend(batch_result)
                    
                    # 중간 체크포인트 저장
                    if (i + 1) % 10 == 0 or i == len(texts) - 1:
                        partial_df = pd.DataFrame(results)
                        if 'orig_index' in partial_df.columns:
                            partial_df.set_index('orig_index', inplace=True)
                        
                        # 안전한 컬럼명 변경
                        new_columns = []
                        for col in partial_df.columns:
                            if col == 'index':
                                new_columns.append(col)  # 'index' 컬럼은 그대로 유지
                            else:
                                new_columns.append(f"{column}_{col}")
                        
                        # 길이 확인 후 할당
                        if len(new_columns) == len(partial_df.columns):
                            partial_df.columns = new_columns
                        else:
                            logger.error(f"Column length mismatch in checkpoint: {len(partial_df.columns)} vs {len(new_columns)}")
                            # 개별 컬럼 이름 변경
                            for i, (old_col, new_col) in enumerate(zip(partial_df.columns, new_columns)):
                                if old_col != new_col:  # 이미 변경된 컬럼은 다시 변경하지 않음
                                    partial_df = partial_df.rename(columns={old_col: new_col})
                        
                        self.checkpoint.save_checkpoint(partial_df, column)
                    
            except Exception as e:
                logger.error(f"인덱스 {idx}의 텍스트 처리 실패: {str(e)}")
                results.append({"index": idx, "orig_index": orig_idx})

        return results

    @retry(stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _process_with_retry(self, classifier, batch_texts: List[str], 
                            batch_indices: List[int], batch_orig_indices: List[str]) -> List[Dict]:
        """비동기로 API 호출 및 재시도"""
        results = []

        for i, (text, batch_idx, orig_idx) in enumerate(zip(batch_texts, batch_indices, batch_orig_indices)):
            async with self.semaphore:
                try:
                    # 한 번에 하나의 텍스트 처리 - classifier는 비동기 함수여야 함
                    single_result = await classifier([text], self.semaphore)
                    
                    # 결과 확인 및 로깅
                    if single_result and len(single_result) > 0:
                        result_item = single_result[0]
                        
                        # 안전하게 결과 병합 (명시적인 딕셔너리 생성)
                        merged_result = {"index": batch_idx, "orig_index": orig_idx}
                        
                        # result_item이 딕셔너리인 경우에만 병합
                        if isinstance(result_item, dict):
                            for key, value in result_item.items():
                                merged_result[key] = value
                        
                        results.append(merged_result)
                    else:
                        # API가 빈 또는 유효하지 않은 결과를 반환한 경우 처리
                        results.append({"index": batch_idx, "orig_index": orig_idx})
                    
                    # 각 개별 결과 로깅
                    logger.info(f"텍스트 {i} (인덱스 {batch_idx}, 원본 인덱스 {orig_idx}) 처리 완료")
                    
                except Exception as e:
                    logger.error(f"텍스트 {i} 처리 중 오류 발생: {str(e)}")
                    # 오류 발생 시에도 인덱스 정보는 유지
                    results.append({"index": batch_idx, "orig_index": orig_idx})
        
        return results

    def _cleanup_checkpoint(self, column: str) -> None:
        """성공적인 처리 후 체크포인트 정리"""
        try:
            checkpoint_path = self.checkpoint.get_checkpoint_path(column)
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
                logger.info(f"Checkpoint cleaned up for {column}")
        except Exception as e:
            logger.error(f"Failed to cleanup checkpoint for {column}: {str(e)}")

    
    #o1-mini 모델 사용
    @retry(stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
        """API 호출 메소드 (직접 API 호출)"""
        try:
            async with semaphore:
                import httpx
                
                # API 요청 준비
                url = "https://api.openai.com/v1/chat/completions"
                headers = {
                    "Authorization": f"Bearer {self.config.API_KEY}",
                    "Content-Type": "application/json"
                }
                data = {
                    "model": "gpt-3.5-turbo",
                    "messages": [
                        {"role": "system", "content": "JSON 형식으로 응답하세요."},
                        {"role": "user", "content": prompt}
                    ],
                    "max_tokens": self.config.MAX_TOKENS,
                    "temperature": self.config.TEMPERATURE
                }
                
                # API 호출
                async with httpx.AsyncClient() as client:
                    response = await client.post(url, headers=headers, json=data, timeout=60.0)
                    response.raise_for_status()  # 오류 발생 시 예외 발생
                    response_data = response.json()
                
                # 응답 처리
                content = response_data["choices"][0]["message"]["content"]
                logger.debug(f"API Response: {content[:200]}...")
                
                # 토큰 사용량 기록
                total_tokens = response_data["usage"]["total_tokens"]
                self.rate_limiter.record_request(total_tokens)
                
                # JSON 파싱
                result = self._validate_and_parse_json(content)
                if not result:
                    raise ValueError("Invalid JSON structure")
                return result

        except httpx.HTTPStatusError as e:
            logger.error(f"HTTP error: {e.response.status_code} - {e.response.text}")
            raise
        except Exception as e:
            logger.error(f"API call failed: {str(e)}")
            raise

    def _validate_and_parse_json(self, content: str) -> List[Dict]:
        """향상된 JSON 응답 검증 및 파싱"""
        try:
            # 원시 내용 로깅
            logger.info(f"원시 API 응답: {content[:500]}...")
            
            # 먼저 직접 JSON 파싱 시도
            try:
                parsed = json.loads(content)
                if isinstance(parsed, list):
                    return parsed
            except json.JSONDecodeError:
                pass  # 직접 파싱이 실패하면 정규식 추출로 계속 진행
            
            # 정규식으로 JSON 추출
            json_pattern = r'```json\s*([\s\S]*?)\s*```|(\[[\s\S]*\])'
            matches = re.findall(json_pattern, content)
            
            for match in matches:
                # 각 일치 항목 시도
                for m in match:
                    if not m.strip():
                        continue
                        
                    try:
                        parsed = json.loads(m.strip())
                        if isinstance(parsed, list):
                            logger.info(f"JSON 파싱 성공: {parsed}")
                            return parsed
                    except:
                        continue
            
            # 마지막 수단: JSON 객체나 배열처럼 보이는 것 찾기
            fallback_pattern = r'(\{[\s\S]*?\}|\[[\s\S]*?\])'
            fallback_matches = re.findall(fallback_pattern, content)
            
            for m in fallback_matches:
                try:
                    parsed = json.loads(m.strip())
                    if isinstance(parsed, dict):
                        # 단일 객체를 목록으로 변환
                        return [parsed]
                    elif isinstance(parsed, list):
                        return parsed
                except:
                    continue
                    
            logger.error(f"응답에서 JSON을 파싱하지 못함")
            return []
            
        except Exception as e:
            logger.error(f"JSON 파싱 오류: {str(e)}")
            return []
        
    async def _classify_cc(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """Chief Complaints 분류"""
        # CC를 3개의 작은 프롬프트로 분할
        cc_results = await self._make_api_call(CCPrompts.cc_analysis_prompt(texts), semaphore)
        history_results = await self._make_api_call(CCPrompts.cc_history_prompt(texts), semaphore)
        severity_results = await self._make_api_call(CCPrompts.cc_severity_prompt(texts), semaphore)
        
        # 결과 병합
        combined_results = []
        for i in range(len(texts)):
            combined_dict = {}
            if i < len(cc_results):
                combined_dict.update(cc_results[i])
            if i < len(history_results):
                combined_dict.update(history_results[i])
            if i < len(severity_results):
                combined_dict.update(severity_results[i])
            
            combined_results.append(combined_dict)
        
        return combined_results
    
    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """약물 복용 분류"""
        results = await self._make_api_call(TreatmentPrompts.medication_prompt(texts), semaphore)
        logger.debug(f"약물 분류 결과: {results}")
        return results

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """장치 사용 분류"""
        results = await self._make_api_call(TreatmentPrompts.device_prompt(texts), semaphore)
        logger.debug(f"장치 분류 결과: {results}")
        return results

    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """습관 분류"""
        results = await self._make_api_call(TreatmentPrompts.habit_prompt(texts), semaphore)
        logger.debug(f"습관 분류 결과: {results}")
        return results

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """찜질 분류"""
        results = await self._make_api_call(TherapyPrompts.hot_pack_prompt(texts), semaphore)
        logger.debug(f"찜질 분류 결과: {results}")
        return results
    
    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """마사지 및 스트레칭 분류"""
        results = await self._make_api_call(TherapyPrompts.massage_prompt(texts), semaphore)
        logger.debug(f"마사지 분류 결과: {results}")
        return results 
        
    async def _classify_present_illness(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """PI 분류 개선"""
        # 유효하지 않은 텍스트 필터링 (None, 비어있거나 길이가 0)
        valid_texts = []
        valid_indices = []
        result_mapping = {}
        
        for i, text in enumerate(texts):
            if text is None or len(str(text).strip()) == 0:
                # 유효하지 않은 텍스트는 빈 결과로 매핑
                result_mapping[i] = {}
            else:
                # 텍스트 길이 제한 (10,000자)
                valid_text = text[:10000] if len(text) > 10000 else text
                valid_texts.append(valid_text)
                valid_indices.append(i)
        
        # 유효한 텍스트가 없으면 빈 결과 반환
        if not valid_texts:
            logger.info("No valid texts for PI classification")
            return [{} for _ in range(len(texts))]
        
        # 로깅 추가: 유효한 텍스트 확인
        for i, text in enumerate(valid_texts):
            logger.info(f"Valid text {i}, length {len(text)}, first 100 chars: {text[:100]}")
        
        smaller_batch_size = 10  # PI에 대해 더 작은 배치 크기 사용
        all_valid_results = []
        
        for i in range(0, len(valid_texts), smaller_batch_size):
            batch = valid_texts[i:i+smaller_batch_size]
            batch_results = []  # 각 배치의 결과를 저장할 리스트
            
            # 로깅 추가: 배치 크기 및 첫 번째 텍스트 확인
            logger.info(f"Processing batch {i//smaller_batch_size + 1}, size: {len(batch)}")
            if batch:
                logger.info(f"First item in batch, length {len(batch[0])}, first 100 chars: {batch[0][:100]}")
        
            try:
                logger.info(f"Processing PI basic info batch {i//smaller_batch_size + 1}")
                pi_aggravating_factors_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_aggravating_factors(batch), semaphore)
                
                logger.info(f"Processing PI examination batch {i//smaller_batch_size + 1}")
                pi_TMJ_PI_desc_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_TMJ_PI_desc(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_TMJ_PI_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_TMJ_PI_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_drug_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_drug_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_closing_dentalgear_desc_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_closing_dentalgear_desc(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_PI_check_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_PI_check(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_PI_diagnosis_jojint_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_PI_diagnosis_jojint(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_occlusal_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_occlusal_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_medication_prescription_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_medication_prescription(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_other_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_other_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_onset_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_onset(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_pattern_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_pattern(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_status_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_status(batch), semaphore)
                
                for j in range(len(batch)):
                    combined_dict = {}
                    if j < len(pi_aggravating_factors_results):
                        combined_dict.update(pi_aggravating_factors_results[j])
                    if j < len(pi_TMJ_PI_desc_results):
                        combined_dict.update(pi_TMJ_PI_desc_results[j])
                    if j < len(pi_TMJ_PI_treatment_results):
                        combined_dict.update(pi_TMJ_PI_treatment_results[j])
                    if j < len(pi_drug_treatment_results):
                        combined_dict.update(pi_drug_treatment_results[j])
                    if j < len(pi_closing_dentalgear_desc_results):
                        combined_dict.update(pi_closing_dentalgear_desc_results[j])
                    if j < len(pi_PI_check_results):
                        combined_dict.update(pi_PI_check_results[j])
                    if j < len(pi_PI_diagnosis_jojint_results):
                        combined_dict.update(pi_PI_diagnosis_jojint_results[j])
                    if j < len(pi_occlusal_treatment_results):
                        combined_dict.update(pi_occlusal_treatment_results[j])
                    if j < len(pi_medication_prescription_results):
                        combined_dict.update(pi_medication_prescription_results[j])
                    if j < len(pi_other_treatment_results):
                        combined_dict.update(pi_other_treatment_results[j])
                    if j < len(pi_onset_results):
                        combined_dict.update(pi_onset_results[j])
                    if j < len(pi_pattern_results):
                        combined_dict.update(pi_pattern_results[j])
                    if j < len(pi_status_results):
                        combined_dict.update(pi_status_results[j])
                    
                    batch_results.append(combined_dict)
                
                all_valid_results.extend(batch_results)
            
            except Exception as e:
                logger.error(f"Error processing PI batch {i//smaller_batch_size + 1}: {str(e)}")
                all_valid_results.extend([{} for _ in range(len(batch))])
        
        # 최종 결과 배열 생성 (원래 순서대로)
        # 먼저 빈 결과 딕셔너리로 채움
        final_results = [{} for _ in range(len(texts))]
        
        # 유효한 텍스트에 대한 결과를 올바른 위치에 넣음
        for valid_idx, result in zip(valid_indices, all_valid_results):
            final_results[valid_idx] = result
        
        return final_results

        
#############################################
# 의학 데이터 처리 함수
#############################################

async def process_medical_data(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """Process medical data with comprehensive error handling and logging"""
    classifier = MedicalTextClassifier(api_key)
    start_time = datetime.now()
    logger.info(f"Starting medical data processing at {start_time}")

    try:
        processed_df = await classifier.process_all_columns(df)

        end_time = datetime.now()
        processing_time = end_time - start_time
        total_rows = len(df)
        processed_columns = [col for col in df.columns if col in classifier.classifiers]

        logger.info("=== Processing Summary ===")
        logger.info(f"Total time: {processing_time}")
        logger.info(f"Total rows processed: {total_rows}")
        logger.info(f"Columns processed: {processed_columns}")

        for col in processed_columns:
            total_entries = df[col].notna().sum()
            processed_entries = sum(1 for col_name in processed_df.columns
                                 if col_name.startswith(f"{col}_")
                                 and processed_df[col_name].notna().any())
            success_rate = (processed_entries / total_entries * 100) if total_entries > 0 else 0
            logger.info(f"{col} - Success rate: {success_rate:.2f}%")

        return processed_df

    except Exception as e:
        logger.critical(f"Critical error during medical data processing: {str(e)}")
        raise


#############################################
# 메인 함수
#############################################

#############################################
# 메인 함수
#############################################

import os
import pandas as pd
import asyncio
import logging
from datetime import datetime
import traceback
from tqdm import tqdm
import glob

# 기존 설정, 클래스 등은 그대로 유지

def main():
    """메인 함수"""
    global logger
    logger = setup_logging()
    total_start_time = datetime.now()

    try:
        # 원본 데이터 불러오기
        # df = pd.read_excel('환자데이터.xlsx')  # 실제 데이터 경로로 변경
        original_df = df  # 여기서 df는 이미 로드된 데이터프레임
        
        # 처리할 총 행 수
        total_rows = len(original_df)
        
        # 청크 크기 설정
        chunk_size = 2000
        
        # 결과 파일을 저장할 디렉토리
        output_dir = 'processed_chunks'
        os.makedirs(output_dir, exist_ok=True)
        
        # 실패한 레코드를 저장할 데이터프레임
        failed_records = pd.DataFrame(columns=['index', 'column', 'text', 'error'])
        
        # 생성된 파일 경로 목록
        processed_files = []
        
        # API 키 설정
        api_key = Config.API_KEY
        
        # 이벤트 루프 가져오기
        loop = asyncio.get_event_loop()
        
        # 청크 단위로 처리
        for start_idx in tqdm(range(0, total_rows, chunk_size), desc="데이터 청크 처리"):
            end_idx = min(start_idx + chunk_size, total_rows)
            
            # 현재 청크 추출
            chunk_df = original_df.iloc[start_idx:end_idx].copy()
            
            # 청크 번호 계산
            chunk_num = start_idx // chunk_size + 1
            
            try:
                logger.info(f"청크 {chunk_num} 처리 시작 (행 {start_idx+1}-{end_idx})")
                
                # 현재 시간 기록
                chunk_start_time = datetime.now()
                
                # 메디컬 텍스트 분류기 인스턴스 생성 (각 청크마다 새로운 인스턴스)
                classifier = MedicalTextClassifier(api_key)
                
                # 청크 처리
                processed_chunk = loop.run_until_complete(classifier.process_all_columns(chunk_df))
                
                # 실패한 레코드 확인
                for column in processed_chunk.columns:
                    if column in classifier.classifiers.keys():
                        related_columns = [col for col in processed_chunk.columns if col.startswith(f"{column}_")]
                        
                        if related_columns:  # 파생 컬럼이 존재하는 경우
                            # 유효한 텍스트가 있지만 처리 결과가 없는 행 찾기
                            mask = processed_chunk[column].notna() & processed_chunk[column].astype(str).str.strip().astype(bool)
                            has_data_mask = mask.copy()
                            
                            for related_col in related_columns:
                                has_data_mask = has_data_mask & processed_chunk[related_col].isna()
                            
                            failed_indices = processed_chunk[has_data_mask].index.tolist()
                            
                            for idx in failed_indices:
                                failed_records = pd.concat([failed_records, pd.DataFrame([{
                                    'index': idx,
                                    'column': column,
                                    'text': processed_chunk.loc[idx, column],
                                    'error': '처리 결과 없음'
                                }])], ignore_index=True)
                                logger.warning(f"레코드 처리 실패: 인덱스 {idx}, 컬럼 {column}, 텍스트: {processed_chunk.loc[idx, column][:100]}...")
                
                # 처리 시간 계산
                chunk_end_time = datetime.now()
                chunk_duration = (chunk_end_time - chunk_start_time).total_seconds()
                logger.info(f"청크 {chunk_num} 처리 완료: {chunk_duration:.2f}초 소요 ({(end_idx-start_idx)/chunk_duration:.2f}행/초)")
                
                # 결과 저장 (타입 변환 후)
                for col in processed_chunk.columns:
                    if processed_chunk[col].dtype == 'object':
                        processed_chunk[col] = processed_chunk[col].fillna('').astype(str)
                
                # parquet 파일로 저장
                output_file = os.path.join(output_dir, f'chunk_{chunk_num:04d}.parquet')
                processed_chunk.to_parquet(output_file)
                processed_files.append(output_file)
                logger.info(f"청크 {chunk_num} 결과 저장 완료: {output_file}")
                
            except Exception as e:
                logger.error(f"청크 {chunk_num} 처리 중 오류 발생: {str(e)}")
                logger.error(traceback.format_exc())
                
                # 오류가 발생한 청크의 모든 행을 실패로 기록
                for idx, row in chunk_df.iterrows():
                    for column in row.index:
                        if column in classifier.classifiers and pd.notna(row[column]) and str(row[column]).strip():
                            failed_records = pd.concat([failed_records, pd.DataFrame([{
                                'index': idx,
                                'column': column,
                                'text': row[column],
                                'error': str(e)
                            }])], ignore_index=True)
        
        # 실패한 레코드 저장
        if not failed_records.empty:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            failed_file = os.path.join(output_dir, f'failed_records_{timestamp}.csv')
            failed_records.to_csv(failed_file, index=False, encoding='utf-8-sig')
            logger.info(f"실패한 레코드 {len(failed_records)}개를 {failed_file}에 저장했습니다.")
        
        # 모든 청크 파일 합치기
        logger.info("모든 청크 파일 합치기 시작...")

        # 원본 인덱스 불러오기 - 중요!
        original_indices = original_df.index.tolist()
        logger.info(f"원본 데이터프레임 인덱스: {original_indices[:5]}...")

        # 모든 파일의 컬럼 목록 확인
        all_columns = set()
        for file in processed_files:
            try:
                chunk = pd.read_parquet(file)
                all_columns.update(chunk.columns)
            except Exception as e:
                logger.error(f"파일 열 분석 실패 {file}: {str(e)}")

        # 최종 결과 데이터프레임 초기화 - 원본 인덱스 사용!
        final_df = pd.DataFrame(index=original_indices, columns=list(all_columns))

        # 각 파일 병합
        for file in tqdm(processed_files, desc="파일 병합"):
            try:
                chunk = pd.read_parquet(file)
                if chunk.empty:
                    continue
                    
                # 인덱스 정보 확인
                logger.info(f"파일 {file}의 인덱스: {chunk.index.tolist()[:5]}...")
                
                # 원본 인덱스 매핑 - 가장 중요한 부분!
                mapping = {}
                for i, idx in enumerate(chunk.index):
                    if i < len(original_indices):
                        mapping[idx] = original_indices[i]
                    
                # 각 컬럼 데이터를 원본 인덱스로 매핑
                for col in chunk.columns:
                    if col in final_df.columns:
                        # 각 행의 데이터를 원본 인덱스 위치에 매핑
                        for old_idx, new_idx in mapping.items():
                            if old_idx in chunk.index and pd.notna(chunk.loc[old_idx, col]) and chunk.loc[old_idx, col] != '':
                                final_df.loc[new_idx, col] = chunk.loc[old_idx, col]
                
            except Exception as e:
                logger.error(f"파일 {file} 병합 실패: {str(e)}")
        
        
        # 최종 결과 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        final_output = f'./final_result/processed_medical_data_{timestamp}.parquet'
        final_df.to_parquet(final_output)
        logger.info(f"최종 결과를 {final_output}에 저장했습니다. (총 {len(final_df)}행)")
        
        # CSV 백업 저장
        csv_output = f'./final_result/processed_medical_data_{timestamp}.csv'
        final_df.to_csv(csv_output, index=True, encoding='utf-8-sig')
        logger.info(f"최종 결과를 CSV 백업 {csv_output}에 저장했습니다.")
        
        # 처리 통계 출력
        logger.info("\n=== 처리 결과 요약 ===")
        logger.info(f"총 처리 레코드: {total_rows}")
        logger.info(f"실패한 레코드: {len(failed_records)}")
        success_rate = ((total_rows - len(failed_records)) / total_rows * 100) if total_rows > 0 else 0
        logger.info(f"처리 성공률: {success_rate:.2f}%")
        
        # 파생 컬럼에 대한 통계
        derived_columns = [col for col in final_df.columns if '_' in col and col.split('_')[0] in classifier.classifiers.keys()]
        logger.info(f"생성된 파생 컬럼: {len(derived_columns)}")
        
        for col in derived_columns:
            valid_values = final_df[col].notna().sum()
            if valid_values > 0:
                logger.info(f"  - {col}: {valid_values}개 유효값 ({valid_values/len(final_df)*100:.2f}%)")
        
    except Exception as e:
        logger.error(f"프로그램 실행 중 오류 발생: {str(e)}")
        logger.error(traceback.format_exc())
    finally:
        # 전체 실행 시간 계산
        total_end_time = datetime.now()
        total_duration = (total_end_time - total_start_time).total_seconds()
        hours, remainder = divmod(total_duration, 3600)
        minutes, seconds = divmod(remainder, 60)
        
        logger.info(f"총 실행 시간: {int(hours)}시간 {int(minutes)}분 {seconds:.2f}초")
        if 'total_rows' in locals() and total_duration > 0:
            records_per_second = total_rows / total_duration
            logger.info(f"평균 처리 속도: {records_per_second:.2f}행/초")
        
        logger.info("프로그램 실행 완료")

if __name__ == "__main__":
    main()

데이터 청크 처리:   0%|          | 0/1 [00:00<?, ?it/s]2025-03-23 21:17:08,700 - __main__ - INFO - 청크 1 처리 시작 (행 1-2)
2025-03-23 21:17:08,726 - __main__ - INFO - 원본 DataFrame 인덱스 타입: int64
2025-03-23 21:17:08,726 - __main__ - INFO - 원본 DataFrame 인덱스 샘플: [19274, 19891]
2025-03-23 21:17:08,729 - __main__ - INFO - 리셋 후 DataFrame 인덱스 샘플: [0, 1]
2025-03-23 21:17:08,731 - __main__ - INFO - 인덱스 컬럼 값 샘플: [19274, 19891]
2025-03-23 21:17:08,733 - __main__ - INFO - orig_index 샘플: ['19274', '19891']
2025-03-23 21:17:08,734 - __main__ - INFO - Processing column: CC
2025-03-23 21:17:08,741 - __main__ - INFO - loggggggggggggggg index :: filtered_df:    index      환자번호         날짜  \
0  19274   2201-06 2022-03-29   
1  19891  2201-261 2022-03-14   

                                                  CC    약   장치   습관  \
0  구강내과#5장치ck ,물리치료장치-매일착용, 불편없었어요.온찜질-최근에는 아예 안했...  NaN  NaN  NaN   
1  구강내과#2SS imp 보톡스 원장님 상의 후 결정하려고 내원했어요왼쪽에서 딱 소리...  NaN  NaN  NaN   

                         찜질 마사지, 스트레칭  \
0        

In [82]:
pd.read_parquet("/Users/nam-yeong/git/prj_centum/gpt_word/checkpoints/CC_checkpoint.parquet")

,index,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,CC_treat_plan,CC_severity,CC_vas,CC_duration
orig_index,,,,,,,,,,,,,
0,0,구강내과,통증,전혀 없었어요,,스트레칭-10번씩 거의 매일 했어요,"교정 치료, 보톡스, 물리치료","턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기","음식 섭취 습관, 수면 자세, 이 악물기 등","물리치료, 약물요법, 장치치료, 보톡스 주사",None,None,None
1,1,"왼쪽 어금니, 위 앞니, 아랫니","딱딱한 소리, 입벌릴 때 귓속 통증, 어금니 통증, 앞니 금이, 아랫니 깎임","딱 소리, 귓속 통증, 어금니 통증, 앞니 금이, 아랫니 깎임",움직임 제한 없음,스트레스로 인한 근육 통증,보톡스,"왼쪽에서 딱 소리나고(최근), 양쪽 다 음식먹을때, 입벌릴때 귓속이 아파요. 오른쪽...",질기고 딱딱한거 많이 피하지는 않았고 치아 물지 않으려고 했어요.,보톡스,None,None,None


In [73]:
cc = pd.read_parquet("/Users/nam-yeong/git/prj_centum/gpt_word/final_result/processed_medical_data_20250323_205905.parquet")

In [74]:
cc

,CC_muscle_joint_desc_stress,찜질_frequency,CC_location,CC,CC_clinic_history_desc,날짜,CC_vas,찜질_duration,CC_disable_desc_jaw,CC_duration,CC_dentalHistory_desc,습관,CC_painUncomp_desc_jaw,찜질_method,"마사지, 스트레칭",CC_pain_type,CC_treat_plan,장치,CC_factor_habbit,찜질,찜질_status,CC_severity,PI,약,환자번호
19274,,,,"구강내과#5장치ck ,물리치료장치-매일착용, 불편없었어요.온찜질-최근에는 아예 안했...",,2022-03-29,None,None,,,,,,,,,,,,,None,,*교합변화 가능성 고지12345678 12345678Dr.남윤진료동기능적 교합검사 ...,,2201-06
19891,,,,구강내과#2SS imp 보톡스 원장님 상의 후 결정하려고 내원했어요왼쪽에서 딱 소리...,,2022-03-14,None,None,,,,,,,,,,,,"온찜질: 2번했어요, 스팀타올, 1번 10분",None,,구강건조증 심함전체적으로 치아에 금감12345678 12345678Dr.남윤진료TM...,,2201-261


In [56]:
df.reset_index()

,index,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI,orig_index
0,19274,2201-06,2022-03-29,"구강내과#5장치ck ,물리치료장치-매일착용, 불편없었어요.온찜질-최근에는 아예 안했...",NaN,NaN,NaN,NaN,NaN,*교합변화 가능성 고지12345678 12345678Dr.남윤진료동기능적 교합검사 ...,19274
1,19891,2201-261,2022-03-14,구강내과#2SS imp 보톡스 원장님 상의 후 결정하려고 내원했어요왼쪽에서 딱 소리...,NaN,NaN,NaN,"온찜질: 2번했어요, 스팀타올, 1번 10분",NaN,구강건조증 심함전체적으로 치아에 금감12345678 12345678Dr.남윤진료TM...,19891


In [61]:
cc

,CC_muscle_joint_desc_stress,찜질_frequency,CC_location,CC,CC_clinic_history_desc,날짜,CC_vas,찜질_duration,CC_disable_desc_jaw,CC_duration,CC_dentalHistory_desc,습관,CC_painUncomp_desc_jaw,찜질_method,"마사지, 스트레칭",CC_pain_type,CC_treat_plan,장치,CC_factor_habbit,찜질,찜질_status,CC_severity,PI,약,환자번호
19274,,,,"구강내과#5장치ck ,물리치료장치-매일착용, 불편없었어요.온찜질-최근에는 아예 안했...",,2022-03-29,None,None,,,,,,,,,,,,,None,,*교합변화 가능성 고지12345678 12345678Dr.남윤진료동기능적 교합검사 ...,,2201-06
19891,,,,구강내과#2SS imp 보톡스 원장님 상의 후 결정하려고 내원했어요왼쪽에서 딱 소리...,,2022-03-14,None,None,,,,,,,,,,,,"온찜질: 2번했어요, 스팀타올, 1번 10분",None,,구강건조증 심함전체적으로 치아에 금감12345678 12345678Dr.남윤진료TM...,,2201-261


In [60]:


df.join(cc, how='left')

ValueError: columns overlap but no suffix specified: Index(['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI'], dtype='object')

In [46]:
df

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI,orig_index
19274,2201-06,2022-03-29,"구강내과#5장치ck ,물리치료장치-매일착용, 불편없었어요.온찜질-최근에는 아예 안했...",NaN,NaN,NaN,NaN,NaN,*교합변화 가능성 고지12345678 12345678Dr.남윤진료동기능적 교합검사 ...,19274
19891,2201-261,2022-03-14,구강내과#2SS imp 보톡스 원장님 상의 후 결정하려고 내원했어요왼쪽에서 딱 소리...,NaN,NaN,NaN,"온찜질: 2번했어요, 스팀타올, 1번 10분",NaN,구강건조증 심함전체적으로 치아에 금감12345678 12345678Dr.남윤진료TM...,19891


In [47]:
pd.read_parquet("/Users/nam-yeong/git/prj_centum/gpt_word/checkpoints/CC_checkpoint.parquet")

,index,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,CC_treat_plan,CC_severity,CC_vas,CC_duration
orig_index,,,,,,,,,,,,,
0,0,턱,통증,통증-전혀 없었어요,,스트레칭-10번씩 거의 매일 했어요,물리치료,"턱관절장애 관련 과거 병력, 치료 이력",딱딱하고 질긴거 그냥 먹었어요. 치아끼리 안물려고 노력했어요.,물리치료,NaN,None,None
1,1,"왼쪽 어금니, 위 앞니, 아랫니","딱딱한 소리, 귓속 통증, 어금니 통증, 앞니 금이, 아랫니 깎임","딱 소리, 귓속 통증, 어금니 통증","앞니 금이, 아랫니 깎임",운동 시 더 아픔,보톡스,"왼쪽에서 딱 소리나고(최근), 양쪽 다 음식먹을때, 입벌릴때 귓속이 아파요. 오른쪽...",질기고 딱딱한거 많이 피하지는 않았고 치아 물지 않으려고 했어요.,보톡스,2.0,None,None


In [23]:
sens = pd.read_parquet('processed_medical_data_20250310_023712.parquet')
nums = pd.read_parquet('../../data/centum_data_numeric_cleaned.parquet')

In [24]:
len(nums), len(sens)


(28108, 28108)

In [25]:
nums.columns

Index(['환자번호', '날짜', 'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획',
       'End feel', 'CMO_before', 'CMO_after', 'MMO_before', 'MMO_after',
       'deviation_pattern_type', 'deviation_direction', 'deviation_intensity',
       'Cap.pal_Pain_Intensity', 'Cap.pal_Pain_Direction',
       'Cap.pal_Pain_Situation', 'M.pal_Pain_Intensity',
       'M.pal_Pain_Direction', 'M.pal_Pain_Situation', 'Noise_Code',
       'Noise_Direction', 'Noise_Intensity', 'Noise_Situation',
       'Occlusion_lt_number', 'Occlusion_rt_number', 'Occlusion_lt_Intensity',
       'Occlusion_rt_Intensity', 'oj', 'ob', 'Midline_Shift_Jaw',
       'Midline_Shift_Direction_x', 'Midline_Shift_Direction_y',
       'Midline_Shift_Amount', 'CRCO_Direction_x', 'CRCO_Direction_y',
       'CRCO_Amount', 'Tongue_ridging_Intensity', 'Rt_before', 'Rt_after',
       'Lt_bef

In [26]:
sens = sens[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method']]

In [27]:
fin_df = pd.merge(nums, sens, on=['환자번호','날짜'], how='left')
cols = [
       '환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',
       'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획',
       'End feel',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method',
       'CMO_before', 'CMO_after', 'MMO_before', 'MMO_after',
       'deviation_pattern_type', 'deviation_direction', 'deviation_intensity',
       'Cap.pal_Pain_Intensity', 'Cap.pal_Pain_Direction',
       'Cap.pal_Pain_Situation', 'M.pal_Pain_Intensity',
       'M.pal_Pain_Direction', 'M.pal_Pain_Situation', 'Noise_Code',
       'Noise_Direction', 'Noise_Intensity', 'Noise_Situation',
       'Occlusion_lt_number', 'Occlusion_rt_number', 'Occlusion_lt_Intensity',
       'Occlusion_rt_Intensity', 'oj', 'ob', 'Midline_Shift_Jaw',
       'Midline_Shift_Direction_x', 'Midline_Shift_Direction_y',
       'Midline_Shift_Amount', 'CRCO_Direction_x', 'CRCO_Direction_y',
       'CRCO_Amount', 'Tongue_ridging_Intensity', 'Rt_before', 'Rt_after',
       'Lt_before', 'Lt_after', 'Next_Visit_Days'
]
fin_df = fin_df[cols]
fin_df.head()

fin_df.to_parquet('../../data/final_without_pi_centum_data_with_medical_data.parquet')
fin_df.to_csv('../../data/final_without_pi_centum_data_with_medical_data.csv', index=False, encoding='utf-8-sig')